In [1]:
!nvidia-smi

Wed Jul 29 14:16:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install datasets scikit-learn accelerate -q


In [5]:
!pip install -U datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.0 MB/s eta 0:00:00


In [6]:
#import +load dataset
from datasets import load_dataset

raw = load_dataset("FinanceMTEB/financial_phrasebank")
print(raw)
print(raw["train"][0])


README.md:   0%|          | 0.00/465 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  104kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 80.2kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1264 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 1264
    })
    test: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 1000
    })
})
{'text': 'The Samsung Mobile Applications Store was launched in January 2009 by Samsung Mobile Innovator , a program which enables mobile software developers to create applications for use across Samsung mobile devices .', 'label_text': 'neutral', 'label': 1}


In [7]:
#class imbalance check(to_pandas)
import pandas as pd
import numpy as np


train_df = raw["train"].to_pandas()
test_df = raw["test"].to_pandas()

print("Shape:", train_df.shape)
print("\n First 3 rows:")
print(train_df.head(3))

print("\n Train Class Balance:")
print(train_df["label"].value_counts())

print("\n Class balance (numpy):")
print(np.bincount(train_df["label"].to_numpy()))




Shape: (1264, 3)

 First 3 rows:
                                                text label_text  label
0  The Samsung Mobile Applications Store was laun...    neutral      1
1  F-Secure , a developer of security solutions a...    neutral      1
2  The company serves customers in various indust...    neutral      1

 Train Class Balance:
label
1    779
2    320
0    165
Name: count, dtype: int64

 Class balance (numpy):
[165 779 320]


In [8]:
# data cleaning
def clean_text(series: pd.Series) -> pd.Series:
    return (
        series
        .str.lower()
        .str.replace(r"[^a-z0-9\s]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

train_df["clean_text"] = clean_text(train_df["text"])
test_df["clean_text"]  = clean_text(test_df["text"])

print("Columns now:", train_df.columns.tolist())
print("\nOriginal:", train_df["text"].iloc[0])
print("\nCleaned: ", train_df["clean_text"].iloc[0])

Columns now: ['text', 'label_text', 'label', 'clean_text']

Original: The Samsung Mobile Applications Store was launched in January 2009 by Samsung Mobile Innovator , a program which enables mobile software developers to create applications for use across Samsung mobile devices .

Cleaned:  the samsung mobile applications store was launched in january 2009 by samsung mobile innovator a program which enables mobile software developers to create applications for use across samsung mobile devices


In [21]:
#tokenizationmethod, pandas ds to huggingface ds , text before tokenization
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

train_ds = Dataset.from_pandas(train_df[["clean_text", "label"]])
test_ds = Dataset.from_pandas(test_df[["clean_text", "label"]])

print(" Before Tokenization")
print(train_ds[0])

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

 Before Tokenization
{'clean_text': 'the samsung mobile applications store was launched in january 2009 by samsung mobile innovator a program which enables mobile software developers to create applications for use across samsung mobile devices', 'label': 1}


In [22]:
# CELL 6 - Actual Tokenization
def tokenize_fn(batch):
    return tokenizer(
        batch["clean_text"],
        padding="max_length",  # pad short sentences with [PAD] tokens
        truncation=True,       # cut long sentences at 128 tokens
        max_length=128,        # every sentence = exactly 128 tokens
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

# See before vs after
print("Keys after tokenization:", train_ds[0].keys())
print("\nToken IDs (first 20):", train_ds[0]["input_ids"][:20])
print("Attention mask (first 20):", train_ds[0]["attention_mask"][:20])
print("Label:", train_ds[0]["label"])

Map:   0%|          | 0/1264 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Keys after tokenization: dict_keys(['clean_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])

Token IDs (first 20): [101, 1996, 19102, 4684, 5097, 3573, 2001, 3390, 1999, 2254, 2268, 2011, 19102, 4684, 7601, 7103, 4263, 1037, 2565, 2029]
Attention mask (first 20): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Label: 1


In [23]:
#Class Weights
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_df["label"].to_numpy()
)

print("Class weights:", class_weights)
print(f"  negative (0): {class_weights[0]:.2f}")
print(f"  neutral  (1): {class_weights[1]:.2f}")
print(f"  positive (2): {class_weights[2]:.2f}")

Class weights: [2.55353535 0.54086436 1.31666667]
  negative (0): 2.55
  neutral  (1): 0.54
  positive (2): 1.32


In [24]:
# Save tokenized datasets
train_ds.save_to_disk("train_tokenized")
test_ds.save_to_disk("test_tokenized")
print("Saved!")

Saving the dataset (0/1 shards):   0%|          | 0/1264 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saved!


In [25]:
import os

print(os.path.exists("train_tokenized"))
print(os.path.exists("test_tokenized"))

True
True


In [26]:
from datasets import load_from_disk
train_ds = load_from_disk("train_tokenized")
test_ds = load_from_disk("test_tokenized")
print("Loaded!", train_ds, test_ds)

Loaded! Dataset({
    features: ['clean_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1264
}) Dataset({
    features: ['clean_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1000
})


In [27]:
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=3
)

train_ds_model = train_ds.rename_column("label", "labels")
test_ds_model  = test_ds.rename_column("label", "labels")
train_ds_model.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds_model.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        weights = torch.tensor(class_weights, dtype=torch.float).to(logits.device)
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_model,
    eval_dataset=test_ds_model,
    compute_metrics=compute_metrics,
)

print("Everything ready!")
print(f"Training sentences: {len(train_ds_model)}")
print(f"Testing sentences:  {len(test_ds_model)}")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Everything ready!
Training sentences: 1264
Testing sentences:  1000


In [28]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.275217,0.927000,0.888802
2,No log,0.152355,0.959000,0.938580
3,No log,0.162634,0.963000,0.943917


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=237, training_loss=0.3529235582311445, metrics={'train_runtime': 144.137, 'train_samples_per_second': 26.308, 'train_steps_per_second': 1.644, 'total_flos': 249431520006144.0, 'train_loss': 0.3529235582311445, 'epoch': 3.0})

In [29]:
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

# Detect device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")

def predict(sentence):
    # Step 1: Clean
    clean = sentence.lower()
    clean = ''.join(c for c in clean if c.isalnum() or c.isspace())

    # Step 2: Tokenize
    inputs = tokenizer(
        clean,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=128
    )

    # Step 3: Move to GPU ← THIS WAS MISSING BEFORE
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Step 4: Feed to model
    with torch.no_grad():
        outputs = model(**inputs)

    # Step 5: Get confidence scores
    probs = torch.softmax(outputs.logits, dim=-1)[0]

    # Step 6: Pick highest
    labels = {0: "NEGATIVE 📉", 1: "NEUTRAL 😐", 2: "POSITIVE 📈"}
    pred = torch.argmax(probs).item()

    print(f"Sentence:   {sentence}")
    print(f"Prediction: {labels[pred]}")
    print(f"Confidence: {probs[pred].item()*100:.1f}%")
    print(f"  negative: {probs[0].item()*100:.1f}%")
    print(f"  neutral:  {probs[1].item()*100:.1f}%")
    print(f"  positive: {probs[2].item()*100:.1f}%")
    print("---")

# Test sentences
predict("Apple reported record breaking profits this quarter")
predict("The company filed for bankruptcy today")
predict("Sales remained unchanged from last year")
predict("Operating costs fell sharply boosting margins")

Using device: cuda
Sentence:   Apple reported record breaking profits this quarter
Prediction: POSITIVE 📈
Confidence: 97.7%
  negative: 2.0%
  neutral:  0.3%
  positive: 97.7%
---
Sentence:   The company filed for bankruptcy today
Prediction: NEUTRAL 😐
Confidence: 98.9%
  negative: 0.7%
  neutral:  98.9%
  positive: 0.4%
---
Sentence:   Sales remained unchanged from last year
Prediction: NEGATIVE 📉
Confidence: 82.9%
  negative: 82.9%
  neutral:  5.3%
  positive: 11.7%
---
Sentence:   Operating costs fell sharply boosting margins
Prediction: NEGATIVE 📉
Confidence: 84.9%
  negative: 84.9%
  neutral:  0.2%
  positive: 14.9%
---


In [30]:
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained('/content/drive/MyDrive/sentiment_model')
tokenizer.save_pretrained('/content/drive/MyDrive/sentiment_model')
print("✅ Model saved to Google Drive permanently!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to Google Drive permanently!
